# Etap 1 — Praca z danymi

Budowa wielojęzycznego zbioru SemEval 2023 Task 3 (subtask-3): wykrywanie technik perswazji na poziomie akapitu.

- 6 języków (en, fr, ge, it, po, ru) → ~20 tys. akapitów treningowych
- Puste akapity zostają jako przykłady negatywne (wektor zer)
- Kolumna `lang` → wybór checkpointu na **polskim** dev w Etapie 2
- `pos_weight` (sqrt + clip 10.0) → zapisany dla `BCEWithLogitsLoss`

In [1]:
import sys
sys.modules["torchvision"] = None  # zapobiega konfliktowi VideoReader z datasets

from google.colab import drive
drive.mount("/content/drive")

!pip install -q -U datasets

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.6 MB/s eta 0:00:00


## 1. Ścieżki i stałe

In [2]:
from pathlib import Path
import tarfile

# --- Ścieżki ---
PROJECT_DIR = Path("/content/drive/MyDrive/Uczenie Maszynowe/Project")
DATA_DIR    = PROJECT_DIR / "Data"
BUNDLE_DIR  = DATA_DIR / "SemEval2023"          # tu wyląduje rozpakowany bundle
SEMEVAL_RAW = BUNDLE_DIR / "data"           # folder z podkatalogami en/fr/ge/it/po/ru
TECH_FILE   = BUNDLE_DIR / "scorers" / "techniques_subtask3.txt"
OUT_DIR     = DATA_DIR / "processed"        # wyjście tego etapu
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANGS       = ["en", "fr", "ge", "it", "po", "ru"]
TARGET_LANG = "po"   # język docelowy bota (do selekcji modelu i progów w Etapie 2/3)

assert SEMEVAL_RAW.exists(), f"Nie znaleziono {SEMEVAL_RAW}"
print("Dane SemEval:", SEMEVAL_RAW)


Dane SemEval: /content/drive/MyDrive/Uczenie Maszynowe/Project/Data/SemEval2023/data


## 2. Lista 23 technik

Kolejność klas to **źródło prawdy** dla całego projektu — musi być identyczna w `2_model`, `3_analiza` i bocie.

In [3]:
LABELS = [
    "Appeal_to_Authority",
    "Appeal_to_Popularity",
    "Appeal_to_Values",
    "Appeal_to_Fear-Prejudice",
    "Flag_Waving",
    "Causal_Oversimplification",
    "False_Dilemma-No_Choice",
    "Consequential_Oversimplification",
    "Straw_Man",
    "Red_Herring",
    "Whataboutism",
    "Slogans",
    "Appeal_to_Time",
    "Conversation_Killer",
    "Loaded_Language",
    "Repetition",
    "Exaggeration-Minimisation",
    "Obfuscation-Vagueness-Confusion",
    "Name_Calling-Labeling",
    "Doubt",
    "Guilt_by_Association",
    "Appeal_to_Hypocrisy",
    "Questioning_the_Reputation",
]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
assert len(LABELS) == 23

# Weryfikacja zgodności z plikiem z bundla (kolejność może się różnić — sprawdzamy zbiory)
if TECH_FILE.exists():
    z_pliku = [w.strip() for w in TECH_FILE.read_text(encoding="utf-8").splitlines() if w.strip()]
    assert set(z_pliku) == set(LABELS), "Niezgodność listy technik z plikiem scorers!"
    print("Lista 23 technik zgodna z bundlem.")


Lista 23 technik zgodna z bundlem.


## 3. Parser: złączenie tekstu (`.template`) z etykietami (`.txt`)

Łączymy po kluczu `(article_id, paragraph_id)`. Akapity bez tekstu w template pomijamy.

In [4]:
def wczytaj_jezyk(lang: str, split: str) -> list[dict]:
    base         = SEMEVAL_RAW / lang
    plik_tekst   = base / f"{split}-labels-subtask-3.template"
    plik_etykiet = base / f"{split}-labels-subtask-3.txt"

    teksty = {}
    with open(plik_tekst, encoding="utf-8") as f:
        for linia in f:
            cz = linia.rstrip("\n").rstrip("\r").split("\t")
            if len(cz) >= 3:
                teksty[(cz[0], cz[1])] = cz[2]

    rekordy = []
    with open(plik_etykiet, encoding="utf-8") as f:
        for linia in f:
            cz = linia.rstrip("\n").rstrip("\r").split("\t")
            if len(cz) < 2:
                continue
            aid, pid = cz[0], cz[1]
            etykiety = cz[2].split(",") if len(cz) >= 3 and cz[2].strip() else []
            tekst = teksty.get((aid, pid))
            if tekst is None or not tekst.strip():
                continue
            rekordy.append({
                "article_id":   aid,
                "paragraph_id": int(pid),
                "text":         tekst.strip(),
                "labels_str":   etykiety,
                "lang":         lang,
            })
    return rekordy

# szybki test poprawności parsera
_test = wczytaj_jezyk("po", "train")
print(f"po/train: {len(_test)} akapitów")

po/train: 2294 akapitów


## 4. Budowa splitów + macierz multi-hot

In [5]:
import numpy as np

def multi_hot(labels_str: list[str]) -> list[float]:
    v = np.zeros(len(LABELS), dtype=np.float32)
    for l in labels_str:
        v[label2id[l]] = 1.0
    return v.tolist()

def zbuduj_split(split: str) -> list[dict]:
    rekordy = []
    for lang in LANGS:
        rekordy.extend(wczytaj_jezyk(lang, split))
    for r in rekordy:
        r["labels"] = multi_hot(r["labels_str"])
    return rekordy

train_rek = zbuduj_split("train")
dev_rek   = zbuduj_split("dev")
print(f"train: {len(train_rek)} akapitów | dev: {len(dev_rek)} akapitów")


train: 19901 akapitów | dev: 6456 akapitów


## 5. EDA / kontrola jakości

In [6]:
import pandas as pd

df_tr  = pd.DataFrame(train_rek)
df_dev = pd.DataFrame(dev_rek)
df_tr["n_lab"]  = df_tr["labels_str"].apply(len)
df_dev["n_lab"] = df_dev["labels_str"].apply(len)

print("=== Akapity per język (train) ===")
print(df_tr.groupby("lang").size().to_string())

print(f"\nPuste (bez techniki) w train: {100*(df_tr.n_lab==0).mean():.1f}%")
print(f"Puste (bez techniki) w dev:   {100*(df_dev.n_lab==0).mean():.1f}%")
print(f"Śr. liczba technik na oznaczony akapit (train): "
      f"{df_tr.loc[df_tr.n_lab>0,'n_lab'].mean():.2f}")

print("\n=== Rozkład klas (train, malejąco) ===")
Y_tr = np.array(df_tr["labels"].tolist())
for l, c in sorted(zip(LABELS, Y_tr.sum(0)), key=lambda x: -x[1]):
    print(f"{int(c):6d}  {l}")


=== Akapity per język (train) ===
lang
en    9498
fr    2196
ge    1484
it    2552
po    2294
ru    1877

Puste (bez techniki) w train: 46.1%
Puste (bez techniki) w dev:   49.0%
Śr. liczba technik na oznaczony akapit (train): 2.02

=== Rozkład klas (train, malejąco) ===
  4793  Loaded_Language
  3365  Name_Calling-Labeling
  2781  Doubt
  1477  Questioning_the_Reputation
  1253  Exaggeration-Minimisation
  1139  Appeal_to_Fear-Prejudice
   747  Repetition
   692  Conversation_Killer
   647  Appeal_to_Hypocrisy
   571  Appeal_to_Authority
   535  Slogans
   530  Flag_Waving
   476  Guilt_by_Association
   470  Causal_Oversimplification
   446  Appeal_to_Values
   331  False_Dilemma-No_Choice
   266  Consequential_Oversimplification
   266  Obfuscation-Vagueness-Confusion
   249  Straw_Man
   234  Appeal_to_Popularity
   166  Red_Herring
   120  Appeal_to_Time
   114  Whataboutism


## 6. `pos_weight` dla `BCEWithLogitsLoss`

`sqrt(neg/pos)` z twardym limitem 10.0 — kompromis między wykrywaniem rzadkich klas a szumem. Liczone tylko z train.

In [ ]:
n   = len(Y_tr)
pos = Y_tr.sum(0)
neg = n - pos
pos_weight = np.sqrt(neg / np.clip(pos, 1, None))   # łagodniejsze niż surowe neg/pos
pos_weight = np.clip(pos_weight, a_min=None, a_max=10.0).astype(np.float32)

print("pos_weight (per klasa):")
for l, w in sorted(zip(LABELS, pos_weight), key=lambda x: -x[1]):
    print(f"  {w:5.2f}  {l}")


pos_weight (per klasa):
  10.00  Red_Herring
  10.00  Whataboutism
  10.00  Appeal_to_Time
   9.17  Appeal_to_Popularity
   8.88  Straw_Man
   8.59  Consequential_Oversimplification
   8.59  Obfuscation-Vagueness-Confusion
   7.69  False_Dilemma-No_Choice
   6.60  Appeal_to_Values
   6.43  Causal_Oversimplification
   6.39  Guilt_by_Association
   6.05  Flag_Waving
   6.02  Slogans
   5.82  Appeal_to_Authority
   5.46  Appeal_to_Hypocrisy
   5.27  Conversation_Killer
   5.06  Repetition
   4.06  Appeal_to_Fear-Prejudice
   3.86  Exaggeration-Minimisation
   3.53  Questioning_the_Reputation
   2.48  Doubt
   2.22  Name_Calling-Labeling
   1.78  Loaded_Language


## 7. Zapis: `DatasetDict` + lista klas + `pos_weight`

In [ ]:
from datasets import Dataset, DatasetDict

KOLUMNY = ["text", "labels", "lang", "article_id", "paragraph_id"]
ds = DatasetDict({
    "train":      Dataset.from_pandas(df_tr[KOLUMNY],  preserve_index=False),
    "validation": Dataset.from_pandas(df_dev[KOLUMNY], preserve_index=False),
})

SAVE_PATH = OUT_DIR / "semeval_multilang"
ds.save_to_disk(str(SAVE_PATH))

# źródła prawdy dla Etapu 2/3 i bota
(OUT_DIR / "labels.txt").write_text("\n".join(LABELS), encoding="utf-8")
np.save(OUT_DIR / "pos_weight.npy", pos_weight)

print("Zapisano:")
print(" -", SAVE_PATH)
print(" -", OUT_DIR / "labels.txt")
print(" -", OUT_DIR / "pos_weight.npy")
print(ds)


Saving the dataset (0/1 shards):   0%|          | 0/19901 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6456 [00:00<?, ? examples/s]

Zapisano:
 - /content/drive/MyDrive/Uczenie Maszynowe/Project/Data/processed/semeval_multilang
 - /content/drive/MyDrive/Uczenie Maszynowe/Project/Data/processed/labels.txt
 - /content/drive/MyDrive/Uczenie Maszynowe/Project/Data/processed/pos_weight.npy
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'lang', 'article_id', 'paragraph_id'],
        num_rows: 19901
    })
    validation: Dataset({
        features: ['text', 'labels', 'lang', 'article_id', 'paragraph_id'],
        num_rows: 6456
    })
})


## 8. Kontrola: wczytaj zapisany zbiór i obejrzyj przykłady

In [9]:
from datasets import load_from_disk

ds2 = load_from_disk(str(SAVE_PATH))
print(ds2)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'lang', 'article_id', 'paragraph_id'],
        num_rows: 19901
    })
    validation: Dataset({
        features: ['text', 'labels', 'lang', 'article_id', 'paragraph_id'],
        num_rows: 6456
    })
})


---
## Co dalej — Etap 2

`2_model.ipynb` wczyta zbiór przez `load_from_disk` i przeprowadzi fine-tuning XLM-RoBERTa z `BCEWithLogitsLoss` + `pos_weight`. Checkpoint selekcjonowany na polskim dev (`f1_micro`).